### This notebook focuses on preparing the raw airline operations data for analysis. It includes loading multi-year datasets, creating completed flight and disruption indicators, and aggregating flight-level data to a carrier–airport–month level. The final output is a structured monthly dataset with operational and percentage-based metrics used for subsequent analysis.

# Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
from pathlib import Path


# Define the raw data folder

In [3]:
# Define the folder that contains all raw BTS zip files
raw_data_path = Path("../data/raw")

# Display all files in the raw data folder
list(raw_data_path.glob("*.zip"))

[WindowsPath('../data/raw/bts_ontime_2023_april.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_august.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_december.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_february.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_january.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_july.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_june.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_march.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_may.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_november.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_october.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_september.zip'),
 WindowsPath('../data/raw/bts_ontime_2024_april.zip'),
 WindowsPath('../data/raw/bts_ontime_2024_august.zip'),
 WindowsPath('../data/raw/bts_ontime_2024_december.zip'),
 WindowsPath('../data/raw/bts_ontime_2024_february.zip'),
 WindowsPath('../data/raw/bts_ontime_2024_january.zip'),
 WindowsPath('../data/raw/bts_ontime_2024_

# Create a list of zip files

In [4]:
# Create a sorted list of all zip files in the raw data folder
zip_files = sorted(raw_data_path.glob("*.zip"))

# Print the number of zip files found
print(f"Number of zip files found: {len(zip_files)}")

# Preview the zip files
zip_files

Number of zip files found: 36


[WindowsPath('../data/raw/bts_ontime_2023_april.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_august.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_december.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_february.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_january.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_july.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_june.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_march.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_may.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_november.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_october.zip'),
 WindowsPath('../data/raw/bts_ontime_2023_september.zip'),
 WindowsPath('../data/raw/bts_ontime_2024_april.zip'),
 WindowsPath('../data/raw/bts_ontime_2024_august.zip'),
 WindowsPath('../data/raw/bts_ontime_2024_december.zip'),
 WindowsPath('../data/raw/bts_ontime_2024_february.zip'),
 WindowsPath('../data/raw/bts_ontime_2024_january.zip'),
 WindowsPath('../data/raw/bts_ontime_2024_

# Build a loop to read all zip files

In [6]:
# Create an empty list to store monthly dataframes
monthly_dfs = []

# Create an empty list to store files that fail to load
failed_files = []

# Loop through each zip file
for zip_file in zip_files:
    try:
        # Print the current file being processed
        print(f"Processing file: {zip_file.name}")
        
        # Open the zip file
        with zipfile.ZipFile(zip_file, 'r') as z:
            
            # Get the CSV file name inside the zip
            csv_name = z.namelist()[0]
            
            # Read the CSV into a temporary dataframe
            temp_df = pd.read_csv(z.open(csv_name), low_memory=False)
            
            # Add the source file name as a new column
            temp_df['source_file'] = zip_file.name
            
            # Append the dataframe to the list
            monthly_dfs.append(temp_df)
    
    except Exception as e:
        # Print the error message and store the failed file name
        print(f"Failed to load {zip_file.name}: {e}")
        failed_files.append(zip_file.name)

# Print summary
print(f"\nSuccessfully loaded {len(monthly_dfs)} files.")
print(f"Failed to load {len(failed_files)} files.")
print("Failed files:", failed_files)

Processing file: bts_ontime_2023_april.zip
Processing file: bts_ontime_2023_august.zip
Processing file: bts_ontime_2023_december.zip
Processing file: bts_ontime_2023_february.zip
Processing file: bts_ontime_2023_january.zip
Processing file: bts_ontime_2023_july.zip
Processing file: bts_ontime_2023_june.zip
Processing file: bts_ontime_2023_march.zip
Processing file: bts_ontime_2023_may.zip
Processing file: bts_ontime_2023_november.zip
Processing file: bts_ontime_2023_october.zip
Processing file: bts_ontime_2023_september.zip
Processing file: bts_ontime_2024_april.zip
Processing file: bts_ontime_2024_august.zip
Processing file: bts_ontime_2024_december.zip
Processing file: bts_ontime_2024_february.zip
Processing file: bts_ontime_2024_january.zip
Processing file: bts_ontime_2024_july.zip
Processing file: bts_ontime_2024_june.zip
Processing file: bts_ontime_2024_march.zip
Processing file: bts_ontime_2024_may.zip
Processing file: bts_ontime_2024_november.zip
Processing file: bts_ontime_2024

# Combine all monthly dataframes

In [7]:
# Combine all monthly dataframes into one full dataframe
combined_df = pd.concat(monthly_dfs, ignore_index=True)

# Display the shape of the combined dataset
combined_df.shape

(20928579, 111)

# Preview the combined dataset

In [8]:
combined_df.head()

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109,source_file
0,2023,2,4,26,3,2023-04-26,UA,19977,UA,N851UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bts_ontime_2023_april.zip
1,2023,2,4,26,3,2023-04-26,UA,19977,UA,N37530,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bts_ontime_2023_april.zip
2,2023,2,4,26,3,2023-04-26,UA,19977,UA,N78509,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bts_ontime_2023_april.zip
3,2023,2,4,26,3,2023-04-26,UA,19977,UA,N47280,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bts_ontime_2023_april.zip
4,2023,2,4,26,3,2023-04-26,UA,19977,UA,N826UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bts_ontime_2023_april.zip


# Check the time coverage

## Check the year-month coverage

In [9]:
# Display the unique year-month combinations in a clean sorted format
combined_df[['Year', 'Month']].drop_duplicates().sort_values(['Year', 'Month']).reset_index(drop=True)

,Year,Month
0,2023,1
1,2023,2
2,2023,3
3,2023,4
4,2023,5
5,2023,6
6,2023,7
7,2023,8
8,2023,9
9,2023,10


## Count the number of records per month

In [10]:
# Count the number of flight records in each year-month combination
monthly_record_counts = (
    combined_df
    .groupby(['Year', 'Month'], as_index=False)
    .size()
    .rename(columns={'size': 'record_count'})
)

# Display the monthly record counts
monthly_record_counts

,Year,Month,record_count
0,2023,1,538837
1,2023,2,502749
2,2023,3,580322
3,2023,4,561441
4,2023,5,579958
5,2023,6,577262
6,2023,7,601866
7,2023,8,602987
8,2023,9,569338
9,2023,10,598968


# Number of rows per source file

In [11]:
# Count the number of rows loaded from each source file
combined_df['source_file'].value_counts().sort_index()

source_file
bts_ontime_2023_april.zip        561441
bts_ontime_2023_august.zip       602987
bts_ontime_2023_december.zip     570394
bts_ontime_2023_february.zip     502749
bts_ontime_2023_january.zip      538837
bts_ontime_2023_july.zip         601866
bts_ontime_2023_june.zip         577262
bts_ontime_2023_march.zip        580322
bts_ontime_2023_may.zip          579958
bts_ontime_2023_november.zip     563777
bts_ontime_2023_october.zip      598968
bts_ontime_2023_september.zip    569338
bts_ontime_2024_april.zip        582185
bts_ontime_2024_august.zip       619025
bts_ontime_2024_december.zip     590581
bts_ontime_2024_february.zip     519221
bts_ontime_2024_january.zip      547271
bts_ontime_2024_july.zip         634613
bts_ontime_2024_june.zip         611132
bts_ontime_2024_march.zip        591767
bts_ontime_2024_may.zip          609743
bts_ontime_2024_november.zip     575404
bts_ontime_2024_october.zip      615497
bts_ontime_2024_september.zip    582622
bts_ontime_2025_april.zip   

# Save the combined raw dataset

In [42]:
# Save the combined raw dataset to the processed data folder
combined_df.to_parquet("../data/processed/combined_bts_ontime_data.parquet", index=False)

# Print confirmation message
print("Combined raw dataset saved successfully.")

Combined raw dataset saved successfully.


# Create a reduced working dataset from the combined data

## Select relevant columns

In [43]:
# Select the main columns needed for the project
working_df = combined_df[
    [
        'Year',
        'Month',
        'FlightDate',
        'Reporting_Airline',
        'Origin',
        'Dest',
        'DepDelay',
        'ArrDelay',
        'Cancelled',
        'CancellationCode',
        'Diverted',
        'CarrierDelay',
        'WeatherDelay',
        'NASDelay',
        'SecurityDelay',
        'LateAircraftDelay'
    ]
].copy()

# Display the shape of the reduced working dataset
working_df.shape

(20928579, 16)

In [44]:
working_df.head()

,Year,Month,FlightDate,Reporting_Airline,Origin,Dest,DepDelay,ArrDelay,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2023,4,2023-04-26,UA,DFW,EWR,7.0,13.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN
1,2023,4,2023-04-26,UA,DEN,OMA,39.0,47.0,0.0,NaN,0.0,39.0,0.0,8.0,0.0,0.0
2,2023,4,2023-04-26,UA,LAX,CLE,-9.0,-11.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN
3,2023,4,2023-04-26,UA,BOS,IAD,-3.0,-11.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN
4,2023,4,2023-04-26,UA,SEA,DEN,-3.0,-6.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN


# Check data types and missing values

In [45]:
# Display dataset information including data types and non-null counts
working_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20928579 entries, 0 to 20928578
Data columns (total 16 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Year               int64  
 1   Month              int64  
 2   FlightDate         object 
 3   Reporting_Airline  object 
 4   Origin             object 
 5   Dest               object 
 6   DepDelay           float64
 7   ArrDelay           float64
 8   Cancelled          float64
 9   CancellationCode   object 
 10  Diverted           float64
 11  CarrierDelay       float64
 12  WeatherDelay       float64
 13  NASDelay           float64
 14  SecurityDelay      float64
 15  LateAircraftDelay  float64
dtypes: float64(9), int64(2), object(5)
memory usage: 2.5+ GB


In [46]:
# Count missing values in each column of the reduced dataset
working_df.isnull().sum()

Year                        0
Month                       0
FlightDate                  0
Reporting_Airline           0
Origin                      0
Dest                        0
DepDelay               276090
ArrDelay               340445
Cancelled                   0
CancellationCode     20641445
Diverted                    0
CarrierDelay         16557279
WeatherDelay         16557279
NASDelay             16557279
SecurityDelay        16557279
LateAircraftDelay    16557279
dtype: int64

# Convert flight date to datetime

In [47]:
# Convert FlightDate to datetime format
working_df['FlightDate'] = pd.to_datetime(working_df['FlightDate'])

# Confirm the conversion
working_df['FlightDate'].dtype

dtype('<M8[ns]')

# Create a year-month variable

In [48]:
# Create a month-level variable from the flight date
working_df['YearMonth'] = working_df['FlightDate'].dt.to_period('M').astype(str)

# Preview the date-related columns
working_df[['FlightDate', 'YearMonth']].head()

,FlightDate,YearMonth
0,2023-04-26,2023-04
1,2023-04-26,2023-04
2,2023-04-26,2023-04
3,2023-04-26,2023-04
4,2023-04-26,2023-04


# Save the reduced working dataset

In [99]:
# Save the reduced working dataset to the processed data folder
working_df.to_parquet("../data/processed/working_flight_data.parquet", index=False)

# Print confirmation message
print("Working dataset for 2023, 2024 and 2025 saved successfully.")

Working dataset for 2023, 2024 and 2025 saved successfully.


# Check Flight Status and Create the Completed Flights Dataset

# Check cancelled flight distribution

In [51]:
# Count the number of cancelled and non-cancelled flights
working_df['Cancelled'].value_counts(dropna=False)

Cancelled
0.0    20641445
1.0      287134
Name: count, dtype: int64

# Check diverted flight distribution

In [52]:
# Count the number of diverted and non-diverted flights
working_df['Diverted'].value_counts(dropna=False)

Diverted
0.0    20875270
1.0       53309
Name: count, dtype: int64

## Examine Missing Departure Delay by Cancellation Status

In [53]:
# Check how missing departure delay values relate to cancelled flights
working_df.groupby('Cancelled')['DepDelay'].apply(lambda x: x.isnull().sum())

Cancelled
0.0         0
1.0    276090
Name: DepDelay, dtype: int64

## Examine Missing Arrival Delay by Cancellation Status

In [54]:
# Check how missing arrival delay values relate to cancelled flights
working_df.groupby('Cancelled')['ArrDelay'].apply(lambda x: x.isnull().sum())

Cancelled
0.0     53311
1.0    287134
Name: ArrDelay, dtype: int64

## Examine Missing Arrival Delay by Diversion Status

In [55]:
# Check how missing arrival delay values relate to diverted flights
working_df.groupby('Diverted')['ArrDelay'].apply(lambda x: x.isnull().sum())

Diverted
0.0    287136
1.0     53309
Name: ArrDelay, dtype: int64

## View rows with missing delay values

In [56]:
# Display example rows where departure delay or arrival delay is missing
working_df[
    working_df['DepDelay'].isnull() | working_df['ArrDelay'].isnull()
].head(10)

,Year,Month,FlightDate,Reporting_Airline,Origin,Dest,DepDelay,ArrDelay,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,YearMonth
21,2023,4,2023-04-26,UA,MCI,SFO,NaN,NaN,1.0,B,0.0,NaN,NaN,NaN,NaN,NaN,2023-04
334,2023,4,2023-04-26,UA,IAH,SEA,-2.0,NaN,0.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,2023-04
394,2023,4,2023-04-26,UA,EWR,SNA,179.0,NaN,0.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,2023-04
471,2023,4,2023-04-26,UA,EWR,SNA,-2.0,NaN,0.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,2023-04
579,2023,4,2023-04-26,UA,ORD,DFW,-5.0,NaN,0.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,2023-04
593,2023,4,2023-04-26,UA,LAX,EWR,20.0,NaN,0.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,2023-04
620,2023,4,2023-04-26,UA,BDL,DEN,NaN,NaN,1.0,B,0.0,NaN,NaN,NaN,NaN,NaN,2023-04
685,2023,4,2023-04-26,UA,CVG,DEN,NaN,NaN,1.0,B,0.0,NaN,NaN,NaN,NaN,NaN,2023-04
690,2023,4,2023-04-26,UA,DEN,SBA,5.0,NaN,0.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,2023-04
720,2023,4,2023-04-26,UA,SNA,ORD,NaN,NaN,1.0,A,0.0,NaN,NaN,NaN,NaN,NaN,2023-04


## Create the completed flights dataset

In [57]:
# Keep only flights that were not cancelled and not diverted
completed_df = working_df[
    (working_df['Cancelled'] == 0) &
    (working_df['Diverted'] == 0)
].copy()

# Display the shape of the completed flights dataset
completed_df.shape

(20588136, 17)

## Check missing values in delay columns after filtering

In [58]:
# Check for missing values in departure and arrival delay columns
completed_df[['DepDelay', 'ArrDelay']].isnull().sum()

DepDelay    0
ArrDelay    2
dtype: int64

## View the row with missing arrival delay

In [59]:
completed_df[completed_df['ArrDelay'].isnull()]

,Year,Month,FlightDate,Reporting_Airline,Origin,Dest,DepDelay,ArrDelay,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,YearMonth
4966463,2023,5,2023-05-21,YX,LGA,BNA,37.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,2023-05
20776845,2025,9,2025-09-06,YX,JFK,MVY,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,2025-09


## Check whether the flight was truly completed

In [60]:
# Display key status columns for the row with missing arrival delay
completed_df.loc[
    completed_df['ArrDelay'].isnull(),
    ['FlightDate', 'Reporting_Airline', 'Origin', 'Dest', 'Cancelled', 'Diverted', 'DepDelay', 'ArrDelay']
]

,FlightDate,Reporting_Airline,Origin,Dest,Cancelled,Diverted,DepDelay,ArrDelay
4966463,2023-05-21,YX,LGA,BNA,0.0,0.0,37.0,NaN
20776845,2025-09-06,YX,JFK,MVY,0.0,0.0,0.0,NaN


## Check whether actual arrival/departure time columns exist for the missing arrival delay row in the original combined dataset

In [61]:
# Check whether actual and scheduled time columns are available in the full combined dataset
[
    col for col in combined_df.columns
    if col in ['CRSDepTime', 'DepTime', 'CRSArrTime', 'ArrTime', 'WheelsOff', 'WheelsOn']
]

['CRSDepTime', 'DepTime', 'WheelsOff', 'WheelsOn', 'CRSArrTime', 'ArrTime']

In [62]:
# Identify the suspicious row using key identifiers
problem_row = combined_df[
    (combined_df['FlightDate'] == '2025-09-06') &
    (combined_df['Reporting_Airline'] == 'YX') &
    (combined_df['Origin'] == 'JFK') &
    (combined_df['Dest'] == 'MVY') &
    (combined_df['Cancelled'] == 0) &
    (combined_df['Diverted'] == 0) &
    (combined_df['DepDelay'] == 0)
]

# Display more operational timing fields for this row
problem_row[
    [
        'FlightDate',
        'Reporting_Airline',
        'Origin',
        'Dest',
        'Cancelled',
        'Diverted',
        'CRSDepTime',
        'DepTime',
        'CRSArrTime',
        'ArrTime',
        'DepDelay',
        'ArrDelay'
    ]
]

,FlightDate,Reporting_Airline,Origin,Dest,Cancelled,Diverted,CRSDepTime,DepTime,CRSArrTime,ArrTime,DepDelay,ArrDelay
20776845,2025-09-06,YX,JFK,MVY,0.0,0.0,1209,1209.0,1333,NaN,0.0,NaN


## Drop the remaining missing arrival delay row

In [63]:
# Remove the row with missing arrival delay from the completed flights dataset
completed_df = completed_df.dropna(subset=['ArrDelay']).copy()

# Confirm that departure and arrival delay columns are now complete
completed_df[['DepDelay', 'ArrDelay']].isnull().sum()

DepDelay    0
ArrDelay    0
dtype: int64

# Create Delay Indicator Variables

## Create departure delay indicator

In [64]:
# Create a binary flag for departure delays greater than 15 minutes
completed_df['DepDelayed'] = (completed_df['DepDelay'] > 15).astype(int)

# Check the distribution
completed_df['DepDelayed'].value_counts()

DepDelayed
0    16418680
1     4169454
Name: count, dtype: int64

## Create arrival delay indicator

In [65]:
# Create a binary flag for arrival delays greater than 15 minutes
completed_df['ArrDelayed'] = (completed_df['ArrDelay'] > 15).astype(int)

# Check the distribution
completed_df['ArrDelayed'].value_counts()

ArrDelayed
0    16357362
1     4230772
Name: count, dtype: int64

## Create a general flight disruption indicator

In [66]:
# Create a binary flag for flights with either a departure or arrival delay greater than 15 minutes
completed_df['DisruptedFlight'] = (
    (completed_df['Cancelled'] == 1) |
    (completed_df['Diverted'] == 1) |
    (completed_df['DepDelay'] > 15) |
    (completed_df['ArrDelay'] > 15)
).astype(int)


# Check the distribution
completed_df['DisruptedFlight'].value_counts()

DisruptedFlight
0    15557636
1     5030498
Name: count, dtype: int64

## Save the combined flights dataset

In [67]:
# Save the completed flights dataset for later reuse
completed_df.to_parquet("../data/processed/completed_flights_data.parquet", index=False)

# Print confirmation message
print("Completed flights dataset for 2023, 2024 and 2025 saved successfully.")

Completed flights dataset for 2023, 2024 and 2025 saved successfully.


# Aggregate flight-level data to carrier-airport-month level

## Aggregate completed flights

In [68]:
# Aggregate completed flights to carrier-airport-month level
carrier_airport_month_df = (
    completed_df
    .groupby(['YearMonth', 'Reporting_Airline', 'Origin'], as_index=False)
    .agg(
        total_completed_flights=('FlightDate', 'count'),
        dep_delayed_flights=('DepDelayed', 'sum'),
        arr_delayed_flights=('ArrDelayed', 'sum'),
        disrupted_flights=('DisruptedFlight', 'sum'),
        avg_dep_delay=('DepDelay', 'mean'),
        avg_arr_delay=('ArrDelay', 'mean'),
        median_dep_delay=('DepDelay', 'median'),
        median_arr_delay=('ArrDelay', 'median'),
        carrier_delay_minutes=('CarrierDelay', 'sum'),
        weather_delay_minutes=('WeatherDelay', 'sum'),
        nas_delay_minutes=('NASDelay', 'sum'),
        security_delay_minutes=('SecurityDelay', 'sum'),
        late_aircraft_delay_minutes=('LateAircraftDelay', 'sum')
    )
)

# Display the shape of the aggregated dataset
carrier_airport_month_df.shape

(55077, 16)

## Create percentage-based delay metrics

In [69]:
# Create percentage-based delay metrics
carrier_airport_month_df['pct_dep_delayed'] = (
    carrier_airport_month_df['dep_delayed_flights'] /
    carrier_airport_month_df['total_completed_flights']
)

carrier_airport_month_df['pct_arr_delayed'] = (
    carrier_airport_month_df['arr_delayed_flights'] /
    carrier_airport_month_df['total_completed_flights']
)

carrier_airport_month_df['pct_disrupted'] = (
    carrier_airport_month_df['disrupted_flights'] /
    carrier_airport_month_df['total_completed_flights']
)

# Preview key rate columns
carrier_airport_month_df[
    [
        'YearMonth',
        'Reporting_Airline',
        'Origin',
        'total_completed_flights',
        'pct_dep_delayed',
        'pct_arr_delayed',
        'pct_disrupted'
    ]
].head()

,YearMonth,Reporting_Airline,Origin,total_completed_flights,pct_dep_delayed,pct_arr_delayed,pct_disrupted
0,2023-01,9E,ABE,15,0.133333,0.133333,0.133333
1,2023-01,9E,ABY,82,0.146341,0.158537,0.170732
2,2023-01,9E,AEX,58,0.189655,0.172414,0.189655
3,2023-01,9E,AGS,26,0.269231,0.269231,0.269231
4,2023-01,9E,ALB,107,0.196262,0.299065,0.317757


## Aggregate cancellation and diversion information

In [70]:
# Summarise all scheduled flights to capture cancellations and diversions
status_summary_df = (
    working_df
    .groupby(['YearMonth', 'Reporting_Airline', 'Origin'], as_index=False)
    .agg(
        total_scheduled_flights=('FlightDate', 'count'),
        cancelled_flights=('Cancelled', 'sum'),
        diverted_flights=('Diverted', 'sum')
    )
)

# Create cancellation and diversion rates
status_summary_df['pct_cancelled'] = (
    status_summary_df['cancelled_flights'] /
    status_summary_df['total_scheduled_flights']
)

status_summary_df['pct_diverted'] = (
    status_summary_df['diverted_flights'] /
    status_summary_df['total_scheduled_flights']
)

# Preview the status summary
status_summary_df.head()

,YearMonth,Reporting_Airline,Origin,total_scheduled_flights,cancelled_flights,diverted_flights,pct_cancelled,pct_diverted
0,2023-01,9E,ABE,15,0.0,0.0,0.000000,0.0
1,2023-01,9E,ABY,82,0.0,0.0,0.000000,0.0
2,2023-01,9E,AEX,60,2.0,0.0,0.033333,0.0
3,2023-01,9E,AGS,26,0.0,0.0,0.000000,0.0
4,2023-01,9E,ALB,110,3.0,0.0,0.027273,0.0


## Merge delay summary with status summary

In [71]:
# Merge the completed-flight delay summary with cancellation and diversion information
final_monthly_df = carrier_airport_month_df.merge(
    status_summary_df,
    on=['YearMonth', 'Reporting_Airline', 'Origin'],
    how='left'
)

# Sort the final dataset for readability
final_monthly_df = final_monthly_df.sort_values(
    by=['YearMonth', 'Reporting_Airline', 'Origin']
).reset_index(drop=True)

# Display the shape of the final monthly dataset
final_monthly_df.shape

(55077, 24)

## Preview the final monthly dataset

In [72]:
# Display the first five rows of the final monthly analytical dataset
final_monthly_df.head()

,YearMonth,Reporting_Airline,Origin,total_completed_flights,dep_delayed_flights,arr_delayed_flights,disrupted_flights,avg_dep_delay,avg_arr_delay,median_dep_delay,...,security_delay_minutes,late_aircraft_delay_minutes,pct_dep_delayed,pct_arr_delayed,pct_disrupted,total_scheduled_flights,cancelled_flights,diverted_flights,pct_cancelled,pct_diverted
0,2023-01,9E,ABE,15,2,2,2,24.666667,11.533333,-5.0,...,0.0,291.0,0.133333,0.133333,0.133333,15,0.0,0.0,0.000000,0.0
1,2023-01,9E,ABY,82,12,13,14,11.073171,8.500000,-5.0,...,0.0,874.0,0.146341,0.158537,0.170732,82,0.0,0.0,0.000000,0.0
2,2023-01,9E,AEX,58,11,10,11,20.758621,10.827586,-5.0,...,0.0,1217.0,0.189655,0.172414,0.189655,60,2.0,0.0,0.033333,0.0
3,2023-01,9E,AGS,26,7,7,7,24.846154,15.038462,-0.5,...,0.0,113.0,0.269231,0.269231,0.269231,26,0.0,0.0,0.000000,0.0
4,2023-01,9E,ALB,107,21,32,34,8.570093,9.130841,-5.0,...,0.0,827.0,0.196262,0.299065,0.317757,110,3.0,0.0,0.027273,0.0


## Inspect the final monthly dataset

In [73]:
# Display the structure and data types of the final monthly dataset
final_monthly_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55077 entries, 0 to 55076
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   YearMonth                    55077 non-null  object 
 1   Reporting_Airline            55077 non-null  object 
 2   Origin                       55077 non-null  object 
 3   total_completed_flights      55077 non-null  int64  
 4   dep_delayed_flights          55077 non-null  int64  
 5   arr_delayed_flights          55077 non-null  int64  
 6   disrupted_flights            55077 non-null  int64  
 7   avg_dep_delay                55077 non-null  float64
 8   avg_arr_delay                55077 non-null  float64
 9   median_dep_delay             55077 non-null  float64
 10  median_arr_delay             55077 non-null  float64
 11  carrier_delay_minutes        55077 non-null  float64
 12  weather_delay_minutes        55077 non-null  float64
 13  nas_delay_minute

## Check missing values in the final monthly dataset

In [74]:
# Count missing values in each column of the final monthly dataset
final_monthly_df.isnull().sum()

YearMonth                      0
Reporting_Airline              0
Origin                         0
total_completed_flights        0
dep_delayed_flights            0
arr_delayed_flights            0
disrupted_flights              0
avg_dep_delay                  0
avg_arr_delay                  0
median_dep_delay               0
median_arr_delay               0
carrier_delay_minutes          0
weather_delay_minutes          0
nas_delay_minutes              0
security_delay_minutes         0
late_aircraft_delay_minutes    0
pct_dep_delayed                0
pct_arr_delayed                0
pct_disrupted                  0
total_scheduled_flights        0
cancelled_flights              0
diverted_flights               0
pct_cancelled                  0
pct_diverted                   0
dtype: int64

## Save the final monthly dataset

In [75]:
# Save the final carrier-airport-month analytical dataset
final_monthly_df.to_parquet("../data/processed/final_monthly_operational_summary_data.parquet", index=False)

# Print confirmation message
print("Final monthly operational summary for 2023, 2024 and 2025 saved successfully.")

Final monthly operational summary for 2023, 2024 and 2025 saved successfully.
